In [59]:
MODEL_ID = 'moonshotai/kimi-k2-instruct-0905'
EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["GROQ_API_KEY"].startswith("gsk"), \
  "Please specify the GROQ_API_KEY access token in keys.env file"
assert os.environ["HF_TOKEN"].startswith("hf"), \
  "Please specifuy HF_TOKEN environment variable in keys.env file"

In [60]:
import sys
sys.path.append('../06_basic_rag')
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [61]:
!ls ./.cache vector_index

./.cache:
pg3772_3736454afe.txt  pg4204_81e8e90db3.txt

vector_index:
default__vector_store.json  graph_store.json	      index_store.json
docstore.json		    image__vector_store.json


In [62]:
import nest_asyncio
nest_asyncio.apply()

In [63]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Document

import os
import pathlib

INDEX_DIR="vector_index"
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_ID)

Settings.chunk_size = 1024
Settings.chunk_overlap = 20
TOP_K = 2

if os.path.isdir(INDEX_DIR):
    print("Loading in already created index")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)
    index = load_index_from_storage(storage_context)
else:
    gs = gtl.GutenbergSource()
    gs.load_from_url("https://www.gutenberg.org/cache/epub/3772/pg3772.txt")
    gs.load_from_url("https://www.gutenberg.org/cache/epub/4204/pg4204.txt")

    documents = SimpleDirectoryReader(input_dir='./.cache', required_exts=['.txt'], exclude_hidden=False).load_data()

    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=INDEX_DIR)

2026-04-13 18:47:27,645 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-04-13 18:47:31,067 - INFO - 1 prompt is loaded, with the key: query


Loading in already created index


2026-04-13 18:47:32,760 - INFO - Loading all indices.


In [65]:
from llama_index.llms.groq import Groq
from llama_index.core.query_engine import RetrieverQueryEngine

llm = Groq(model=MODEL_ID, api_key = os.environ["GROQ_API_KEY"])
def semantic_rag(question, top_k=TOP_K, verbose=True):
    query_engine = RetrieverQueryEngine.from_args(
        retriever=index.as_retriever(similarity_top_k=top_k), llm = llm,
    )
    response = query_engine.query(question)
    response = {
        "answer": str(response),
        "source_nodes": response.source_nodes
    }
    if verbose:
        print(response['answer'])
        for node in response['source_nodes']:
            print(node)
    return response
semantic_rag("Describe the geology of the Grand Canyon")

2026-04-13 18:48:28,560 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The Grand Canyon is carved into a stack of gently northward-tilted, stratified rocks more than 10,000 ft thick. These layers form a broad staircase of giant treads and risers: each resistant stratum caps a high plateau or mesa, while the softer layers beneath erode back to create sheer, retreating escarpments. The result is a series of immense steps—cliffs often 1,000 ft high and tens of miles across—that descend southward toward the canyon’s rim. Weathering and wind-deflation sculpt the exposed walls into cathedrals, pyramids, towers, arches, and colonnades, leaving isolated buttes and mesas as outliers.
Node ID: d1716778-ffaa-4e85-8edf-d5db5c5f1b1b
Text: Deep residual soils come to  protect all regions of moderate
slope, concealing from view the  rock structure, and the various forms
of the land are due more to  the agencies of erosion and
transportation than to differences in  the resistance of the
underlying rocks.    But in arid regions the mantle is rapidly
removed, even from wel

{'answer': 'The Grand Canyon is carved into a stack of gently northward-tilted, stratified rocks more than 10,000 ft thick. These layers form a broad staircase of giant treads and risers: each resistant stratum caps a high plateau or mesa, while the softer layers beneath erode back to create sheer, retreating escarpments. The result is a series of immense steps—cliffs often 1,000 ft high and tens of miles across—that descend southward toward the canyon’s rim. Weathering and wind-deflation sculpt the exposed walls into cathedrals, pyramids, towers, arches, and colonnades, leaving isolated buttes and mesas as outliers.',
 'source_nodes': [NodeWithScore(node=TextNode(id_='d1716778-ffaa-4e85-8edf-d5db5c5f1b1b', embedding=None, metadata={'file_path': '/home/sharofiddin/practice/py3/gen_ai_design_patterns/10_node_postprocessing/.cache/pg4204_81e8e90db3.txt', 'file_name': 'pg4204_81e8e90db3.txt', 'file_type': 'text/plain', 'file_size': 685889, 'creation_date': '2026-04-13', 'last_modified_dat

In [ ]:
semantic_rag("Describe the geology of Petrified National forest");

2026-04-13 16:25:57,961 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The Petrified Forest region preserves a record of Late Triassic lowlands that once extended westward from the rising Appalachian mountain system.  At that time the area lay near the equator and was crossed by slow, meandering rivers that deposited great thicknesses of reddish sand and silt.  Periodic floods buried vast forests of conifers, cycad-like plants, and giant horsetails (similar to the forty-foot calamites described for the Carboniferous).  The buried trunks were permeated by silica-rich groundwater, which gradually replaced the organic tissue with quartz, creating the brilliantly colored, perfectly preserved petrified logs visible today.  Above these fossil-bearing layers lie younger Triassic strata that record continued river and flood-plain conditions; the entire sequence rests unconformably on the eroded edges of the older, folded Paleozoic rocks that had been uplifted during the Appalachian deformation.
Node ID: d09f101d-1d2c-45bc-a538-4a6cfcb6800c
Text: Although the grea

In [ ]:
semantic_rag("Describe the geology of The Great Chimgan of Tashkent");

2026-04-13 16:28:38,171 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The provided context does not contain any information about the geology of the Great Chimgan of Tashkent.
Node ID: 52989c3c-d0b8-40a3-9eed-ac1f7eb9b7fc
Text: These Scotch metamorphic strata are of gneiss, mica-schist, and
clay-slate of vast thickness, and having a strike from north-east to
south-west almost at right angles to that of the older Laurentian
gneiss before mentioned. The newer crystalline series, comprising the
crystalline rocks of Aberdeenshire, Perthshire, and Forfarshire, were
inf...
Score:  0.719

Node ID: 3050d517-a8f3-4631-aee5-bef5e1fd0b0c
Text: _Auvergne._—The most northern of the fresh-water groups is
situated in  the valley-plain of the Allier, which lies within the
department of the  Puy de Dome, being the tract which went formerly by
the name of the  Limagne d’Auvergne. The average breadth of this tract
is about twenty  miles; and it is for the most part composed of nearly
horizont...
Score:  0.714



In [ ]:
response = semantic_rag("Describe the geology of The Grand Canyon", top_k=4);

2026-04-13 16:44:01,699 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 16:44:02,421 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The Grand Canyon exposes a vast stack of gently northward-tilted sedimentary layers more than 10,000 ft thick. These horizontal strata form gigantic stair-like cliffs and benches; harder beds stand as vertical walls hundreds of feet high, while softer layers retreat to gentler slopes. The canyon’s sides thus descend to the Colorado River in a series of immense steps. Its walls are carved into architectural forms—cathedrals, towers, amphitheaters, arches—by weathering and wind erosion. Talus is swept away almost as fast as it forms, keeping the cliffs bare. Though the canyon is already profound, the youthful river has not yet reached grade; its main task of reducing the high plateau to a low plain and carrying every grain to the sea still lies almost entirely in the future.
Node ID: d1716778-ffaa-4e85-8edf-d5db5c5f1b1b
Text: Deep residual soils come to  protect all regions of moderate
slope, concealing from view the  rock structure, and the various forms
of the land are due more to  the

In [ ]:
from dataclasses import dataclass
import pydantic_ai 
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai import Agent

model = OpenAIChatModel(MODEL_ID)

@dataclass
class Chunk:
    full_text: str
    publication_year: int
    relevant_text: str
    relevance_score: float
def process_node(query, node):
    system_pompt = """
    You will be given query and some text.
    1. Assign a publication year if it's clear from the text, else say it's the current year 
    2. Remove information from the text that is not relevant for answering the question.
    3. Assign a relevance score between 0 and 1 where 1 means that the text answers the question
    """

    agent = Agent(model, output_type=Chunk, system_prompt=system_pompt)
    chunk = agent.run_sync(f"**Query**: {query}\n **Full Text**: {node.text}").output
    if node.metadata['file_name'].startswith("pg4204"):
        chunk.publication_year = 1905
    else :
        chunk.publication_year = 1878
    return chunk
chunks = [process_node(response['source_nodes'], node) for node in response['source_nodes']]


2026-04-13 16:53:00,012 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 16:53:00,218 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 16:53:00,219 - INFO - Retrying request to /openai/v1/chat/completions in 52.000000 seconds
2026-04-13 16:54:14,343 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 16:54:15,181 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 16:54:15,185 - INFO - Retrying request to /openai/v1/chat/completions in 3.000000 seconds
2026-04-13 16:54:21,177 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 16:54:21,576 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 16:54:21,577 - INFO - Retrying re

In [ ]:
chunks[0].relevant_text

'Deep residual soils come to protect all regions of moderate slope, concealing from view the rock structure, and the various forms of the land are due more to the agencies of erosion and transportation than to differences in the resistance of the underlying rocks.\n\nBut in arid regions the mantle is rapidly removed, even from well-nigh level plains and plateaus, by the sweep of the wind and the wash of occasional rains. The geological structure of these regions of naked rock can be read as far as the eye can see, and it is to this structure that the forms of the land are there largely due. In a land mass of horizontal strata, for example, any softer surface rocks wear down to some underlying, resistant stratum, and this for a while forms the surface of a level plateau (Fig. 129). The edges of the capping layer, together with those of any softer layers beneath it, wear back in steep cliffs, dissected by the valleys of wet-weather streams and often swept bare to the base by the wind. As

In [ ]:
latest_year = max([chunk.publication_year for chunk in chunks])

In [ ]:
response['source_nodes']

[NodeWithScore(node=TextNode(id_='d1716778-ffaa-4e85-8edf-d5db5c5f1b1b', embedding=None, metadata={'file_path': '/home/sharofiddin/practice/py3/gen_ai_design_patterns/10_node_postprocessing/.cache/pg4204_81e8e90db3.txt', 'file_name': 'pg4204_81e8e90db3.txt', 'file_type': 'text/plain', 'file_size': 685889, 'creation_date': '2026-04-13', 'last_modified_date': '2026-04-13'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='cfcaa9e9-8899-438c-9caf-c6c6dc8aeba9', node_type='4', metadata={'file_path': '/home/sharofiddin/practice/py3/gen_ai_design_patterns/10_node_postprocessing/.cache/pg4204_81e8e90db3.txt', 'file_name': 'pg4204_81e8e90db3.txt', 'file_type': 'text/plain', 'file_size': 685889, 'creation_date': '202

In [ ]:
def rerank_rag(query, top_k=TOP_K):
    retriever = index.as_retriever(similarity_top_k= top_k*4)
    nodes = retriever.retrieve(query)

    chunks = [process_node(query, node) for node in nodes]
    chunks = sorted(chunks, key = lambda x: x.relevance_score, reverse=True)

    latest_year = max([chunk.publication_year for chunk in chunks])
    chunks = [chunk for chunk in chunks if chunk.publication_year == latest_year]

    chunks = chunks[:top_k]

    system_prompt = """
    Use the information provided in the context to answer the question.
    Limit your answer to what's known based on the provided information.
    """

    agent = Agent(model, output_type=Chunk, system_prompt=system_prompt)
    answer = agent.run_sync(f"**Query**: {query}\n **Context**:{[chunk.relevant_text for chunk in chunks]}\n **Answer**:").output
    response = {
        "answer": answer,
        "source_nodes": chunks
    }
    return response
rerank_rag("Describe the geology of The Grand Canyon", top_k=2)

2026-04-13 17:08:16,275 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:17,944 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:19,693 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:20,717 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:21,226 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-04-13 17:08:22,487 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:24,223 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:08:24,334 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 17:08:24,337 - INFO -

{'answer': Chunk(full_text='The Grand Canyon exposes a very ancient peneplain. Flat-lying Cambrian sandstone sits atop tilted, far older Algonkian rocks. These Algonkian strata total more than 12,000 ft and are divided into two major series: an upper 5,000-ft package of shales, sandstones and limestones, and a lower 7,000-ft section of reddish sandstones with seven or more interbedded lava flows. At the base lies a conglomerate derived from underlying Archean schists, separated from them by a pronounced unconformity that records long episodes of uplift, erosion, and renewed deposition. The Grand Canyon displays a stepped wall of alternating hard cliffs and weaker slopes: horizontal, differently colored strata form giant stair-like ledges hundreds of feet high, carved back by weathering and the down-cutting Colorado River.', publication_year=0, relevant_text='The Grand Canyon exposes a very ancient peneplain. Flat-lying Cambrian sandstone sits atop tilted, far older Algonkian rocks. The

In [ ]:
@dataclass
class DisambiguationResult:
    is_ambiguous: bool
    ambiguous_term: str
    possibility_1: str
    possibility_2: str

def disambiguate(query, node1, node2):
    system_prompt="""
     You will be given a query and two retrieved passages on which to base the answer to the query.
    Respond by saying whether the two passages are referring to two different entities with the same term.
    For example, the query might be about "Red River", and one passage might be about the
    Red River in Minnesota whereas the other might be about the Red River on the Oklahoma/Texas border.
    If there is no ambiguity between the two passages, return False for is_ambiguous.
    """
    agent = Agent(model, output_type=DisambiguationResult, system_prompt=system_prompt)
    return agent.run_sync(f"**Query**: {query}\n **Passage 1**: {node1.text}\n Passage 2**: {node2.text}").output

In [ ]:
def get_nodes(query):
    response = semantic_rag(query, top_k=10, verbose=False)
    return response
response = get_nodes("Name the characteristics of coal-bearing strata in Newcastle")

2026-04-13 17:22:17,893 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:22:18,450 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:22:19,661 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:22:20,200 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 17:22:21,097 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
for node in response['source_nodes']:
    result = disambiguate("Name the characteristics of coal-bearing strata in Newcastle", response['source_nodes'][0], node)
    if result.is_ambiguous:
        print(result)

2026-04-13 17:22:26,396 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-04-13 17:22:26,744 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-04-13 17:22:26,856 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 17:22:26,858 - INFO - Retrying request to /openai/v1/chat/completions in 5.000000 seconds
2026-04-13 17:22:32,703 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-04-13 17:22:32,831 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-13 17:22:32,832 - INFO - Retrying request to /openai/v1/chat/completions in 14.000000 seconds
2026-04-13 17:22:47,486 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-04-13 17:22:47,601